# 03 — Hypothesis Testing

Formal case-vs-control comparisons for each analysis arm, as a check on the correlation
screen in `02` before committing to a regression specification.

- **Environmental:** Mann-Whitney U for each colonization-pressure variable (cases vs.
  controls); non-parametric since CP is right-skewed.
- **Patient:** Mann-Whitney U for antibiotic course counts; chi-square for `any_abx_0_60`.
- Benjamini-Hochberg correction across the many antibiotic-class / CP tests to control the
  false discovery rate.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency
from statsmodels.stats.multitest import multipletests

sys.path.append(str(Path.cwd().parent))
from src.data_loading import load_processed

env = load_processed("environmental_mrsa.csv")
pat = load_processed("patient_mrsa.csv")

## Environmental: colonization pressure, cases vs. controls

In [2]:
cp_cols = [c for c in env.columns if c.endswith("_cp")]

rows = []
cases = env.loc[env["group_binary"] == 1]
controls = env.loc[env["group_binary"] == 0]
for c in cp_cols:
    stat, p = mannwhitneyu(cases[c], controls[c], alternative="two-sided")
    rows.append({
        "variable": c,
        "median_case": cases[c].median(),
        "median_control": controls[c].median(),
        "p_value": p,
    })
env_results = pd.DataFrame(rows)
env_results["p_adj_bh"] = multipletests(env_results["p_value"], method="fdr_bh")[1]
env_results.sort_values("p_adj_bh")

,variable,median_case,median_control,p_value,p_adj_bh
0,DS_Entero_cp,9.885012,10.925777,0.007605,0.068448
3,VSE_cp,2.334266,2.590160,0.033449,0.150520
1,ESBL_cp,2.739494,2.993956,0.092839,0.278517
5,MSSA_cp,3.269927,3.399885,0.276185,0.621416
2,CDiff_cp,0.755034,0.756571,0.680617,0.875079
6,MRSA_cp,1.649825,1.608602,0.531389,0.875079
8,DR_PsA_cp,0.712003,0.738348,0.591255,0.875079
7,DS_PsA_cp,1.711268,1.727959,0.838543,0.943361
4,VRE_cp,0.786459,0.776252,0.979162,0.979162


## Patient: antibiotic exposure, cases vs. controls

In [3]:
abx_cols = [c for c in pat.columns if c.endswith("_0_60") and c != "any_abx_0_60"]

rows = []
cases = pat.loc[pat["group_binary"] == 1]
controls = pat.loc[pat["group_binary"] == 0]
for c in abx_cols:
    stat, p = mannwhitneyu(cases[c], controls[c], alternative="two-sided")
    rows.append({
        "variable": c,
        "median_case": cases[c].median(),
        "median_control": controls[c].median(),
        "p_value": p,
    })
pat_results = pd.DataFrame(rows)
pat_results["p_adj_bh"] = multipletests(pat_results["p_value"], method="fdr_bh")[1]
pat_results.sort_values("p_adj_bh")

,variable,median_case,median_control,p_value,p_adj_bh
7,sulfonamide_0_60,0.0,0.0,0.000153,0.002140
0,penicillin_0_60,0.0,0.0,0.207705,0.702268
2,anti_staph_beta_lactam_0_60,0.0,0.0,0.197575,0.702268
1,extended_spectrum_penicillin_0_60,0.0,0.0,0.451205,0.702268
3,cephalosporin_0_60,0.0,0.0,0.297492,0.702268
4,extended_spectrum_cephalosporin_0_60,0.0,0.0,0.382109,0.702268
10,anti_anaerobe_0_60,0.0,0.0,0.381507,0.702268
9,anti_Cdiff_0_60,0.0,0.0,0.296212,0.702268
13,tetracycline_0_60,0.0,0.0,0.451458,0.702268
6,fluoroquinolone_0_60,0.0,0.0,0.635460,0.808768


In [4]:
ct = pd.crosstab(pat["group"], pat["any_abx_0_60"])
chi2, p, dof, expected = chi2_contingency(ct)
print(ct)
print(f"chi2={chi2:.2f}, p={p:.4g}")

any_abx_0_60     0    1
group                  
case          1014   88
control       2323  333
chi2=15.77, p=7.153e-05


## Summary

Carry the predictors significant after BH correction (from both arms) into the logistic
regression stage as the primary covariates of interest; non-significant ones are still
included if the literature supports them, but are not the focus of interpretation.